# Setup

Run this notebook once before any of the analysis notebooks. It downloads the IQB data to ``data`` folder for the specified date range. 


In [ ]:
import requests
import sys
from pathlib import Path

ROOT_DIR = Path().resolve() 
DATA_CACHE    = ROOT_DIR / 'data'

print(f'Root dir: {ROOT_DIR}')
print(f'Data target : {DATA_CACHE}')

if sys.version_info < (3, 12):
    raise RuntimeError(f'Python 3.12+ required, found {sys.version.split()[0]}')
print('Python version OK')

PULL_START_DATE  = '2024-01-01'
PULL_END_DATE = '2026-06-01'

DATA_CACHE.mkdir(parents=True, exist_ok=True)

MANIFEST_URL = "https://raw.githubusercontent.com/m-lab/iqb/main/data/state/ghremote/manifest.json"

manifest = requests.get(MANIFEST_URL).json()

urls = {
    path: meta['url']
    for path, meta in manifest['files'].items()
    if any(d in path for d in ['downloads_by_country', 'uploads_by_country'])
    and path.endswith('.parquet')
    and PULL_START_DATE <= path.split('/')[2][:10].replace('T','') <= PULL_END_DATE  # start_date filter
}

existing = sum(1 for path in urls if (DATA_CACHE / Path(*Path(path).parts[1:])).exists())
print(f"{existing} / {len(urls)} files already cached, downloading {len(urls) - existing}...")

print(f"Downloading {len(urls)} files...")
downloaded = []

for path, url in urls.items():
    dest = DATA_CACHE / Path(*Path(path).parts[1:])  # strip leading 'cache/'
    dest.parent.mkdir(parents=True, exist_ok=True)
    if dest.exists():
        continue  # skip already cached
    r = requests.get(url)
    dest.write_bytes(r.content)
    downloaded.append(dest)
    if len(downloaded) % 5 == 0:
        print(f"  Downloaded {len(downloaded)} / {len(urls)}")

print("Pull complete")